In [1]:
!pip -q install -U duckdb pyyaml geopandas shapely pyarrow requests

In [2]:
import os

In [3]:
from google.colab import drive

drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [4]:
!ls /content/

drive  sample_data


In [5]:
import subprocess
subprocess.run(["pip", "install", "--upgrade", "certifi"], check=True)

import certifi, os
os.environ["SSL_CERT_FILE"] = certifi.where()
os.environ["CURL_CA_BUNDLE"] = certifi.where()
os.environ["REQUESTS_CA_BUNDLE"] = certifi.where()

In [6]:
import os
os.chdir("/content/drive/MyDrive/Gates Foundation/Building Dataset Validation/")

In [7]:
# from src.vectordownloader import UrbanVectorDownloader

In [8]:
import yaml 
import geopandas as gpd 

BASE = "/content/drive/MyDrive/Gates Foundation/Building Dataset Validation"
CONFIG = f"{BASE}/configs/ghs_obat.yaml"

with open(CONFIG, "r") as f:
    config = yaml.safe_load(f)


def read_aoi(self):
    """Read the AOI (Area of Interest) from the specified path."""

    aoi_path = self.config['aoi']['path']
    crs_out = self.config['aoi']['crs_out']
    buff = float(self.config["aoi"].get("buffer_meters", 0) or 0)

    aoi = gpd.read_file(aoi_path)
    if aoi.crs is None:
        # self.logger.warning("AOI CRS missing; assuming EPSG:4326")
        aoi = aoi.set_crs("EPSG:4326")



    aoi = aoi.to_crs(self.config["aoi"]["crs_out"])

    if buff > 0:
        if aoi.crs is not None and not aoi.crs.is_projected:
            # self.logger.warning("AOI CRS missing; assuming EPSG:4326")
            aoi_metric = aoi.to_crs("EPSG:3857")
            aoi_metric["geometry"] = aoi_metric.geometry.buffer(buff)
            aoi = aoi_metric.to_crs(aoi.crs)
        else:
            aoi["geometry"] = aoi.geometry.buffer(buff)

    # convert to crs_out 
    if str(aoi.crs) != str(crs_out):
        # self.logger.info("Reprojecting AOI | %s -> %s", aoi.crs, crs_out)
        aoi = aoi.to_crs(crs_out)

    # self.logger.info("Successfully loaded AOI  | rows=%d | crs=%s", len(aoi_gdf), aoi_gdf.crs)

    return aoi

In [9]:
from __future__ import annotations

import json
import zipfile
from dataclasses import dataclass
from pathlib import Path
from typing import Dict, List, Optional, Tuple

import duckdb
import geopandas as gpd
import pandas as pd
import requests
import yaml


# ----------------
# IO / config
# ----------------
def load_config(path: str | Path) -> dict:
    path = Path(path)
    with path.open("r") as f:
        return yaml.safe_load(f)


def read_aoi_geojson(aoi_path: str | Path, crs_out: str = "EPSG:4326") -> gpd.GeoDataFrame:
    gdf = gpd.read_file(aoi_path)
    if gdf.crs is None:
        gdf = gdf.set_crs("EPSG:4326")
    return gdf.to_crs(crs_out)


def ensure_dir(p: Path) -> None:
    p.mkdir(parents=True, exist_ok=True)


def ensure_parent_dir(p: Path) -> None:
    p.parent.mkdir(parents=True, exist_ok=True)


def assert_is_dir_path(p: Path, cfg_key: str) -> None:
    if p.suffix:
        raise ValueError(f"{cfg_key} must be a DIRECTORY path, but got a file-like path: {p}")


# ----------------
# ISO3 lookup
# ----------------
def get_iso3_for_row(cfg: dict, row: pd.Series) -> str:
    iso3_cfg = cfg.get("ghs_obat", {}).get("iso3")
    if iso3_cfg:
        return str(iso3_cfg).upper()

    iso3_col = cfg.get("ghs_obat", {}).get("iso3_col")
    if iso3_col and iso3_col in row.index:
        v = row[iso3_col]
        if v is not None and str(v).strip():
            return str(v).upper()

    raise ValueError(
        "No ISO3 available. Set ghs_obat.iso3 in the YAML, or set ghs_obat.iso3_col "
        "to a field present in the AOI file."
    )


# ----------------
# URL index
# ----------------
def load_url_index(url_index_csv: str | Path) -> Dict[str, str]:
    df = pd.read_csv(url_index_csv)
    if "iso3" not in df.columns or "url" not in df.columns:
        raise ValueError(f"url_index_csv must have columns ['iso3','url'] but got: {list(df.columns)}")
    df["iso3"] = df["iso3"].astype(str).str.upper()
    return dict(zip(df["iso3"], df["url"]))


# ----------------
# Download + unzip (CSV-only)
# ----------------
def download_file(url: str, dest: Path, timeout: int = 180, chunk_mb: int = 8) -> Path:
    """
    Stream download with atomic tmp write.
    """
    ensure_parent_dir(dest)
    tmp = dest.with_suffix(dest.suffix + ".part")
    if dest.exists():
        return dest

    with requests.get(url, stream=True, timeout=timeout) as r:
        r.raise_for_status()
        with tmp.open("wb") as f:
            for chunk in r.iter_content(chunk_size=chunk_mb * 1024 * 1024):
                if chunk:
                    f.write(chunk)

    tmp.replace(dest)
    return dest


def unzip_to_dir(zip_path: Path, out_dir: Path) -> None:
    ensure_dir(out_dir)
    with zipfile.ZipFile(zip_path, "r") as z:
        z.extractall(out_dir)


def find_obat_csv(extract_dir: Path, iso3: str) -> Optional[Path]:
    """
    Pick a CSV deterministically from extracted content.
    Preference: contains ISO3 and 'OBAT' in name.
    """
    iso3 = iso3.upper()
    candidates = sorted(extract_dir.rglob("*.csv"))
    if not candidates:
        return None

    def score(p: Path) -> int:
        name = p.name.upper()
        s = 0
        if "OBAT" in name:
            s += 2
        if iso3 in name:
            s += 3
        return s

    candidates.sort(key=score, reverse=True)
    return candidates[0]


def resolve_country_obat_csv(cfg: dict, iso3: str) -> Path:
    """
    Ensures the OBAT country ZIP is downloaded + extracted and returns the extracted CSV path.
    """
    iso3 = iso3.upper()
    ghs_cfg = cfg["ghs_obat"]

    local_dir = Path(ghs_cfg["local_dir"])
    assert_is_dir_path(local_dir, "ghs_obat.local_dir")
    ensure_dir(local_dir)

    # extraction dir (keeps things tidy)
    extract_dir = local_dir / f"iso3={iso3}"
    ensure_dir(extract_dir)

    # If CSV already exists, reuse
    csvp = find_obat_csv(extract_dir, iso3=iso3)
    if csvp and csvp.exists():
        return csvp

    # Else download ZIP via url_index
    url_index_csv = ghs_cfg.get("url_index_csv")
    if not url_index_csv:
        raise FileNotFoundError("ghs_obat.url_index_csv is required for CSV downloads (iso3,url).")

    url_map = load_url_index(url_index_csv)
    if iso3 not in url_map:
        raise KeyError(f"No download URL for ISO3={iso3} in {url_index_csv}")

    url = url_map[iso3]
    fn = url.split("/")[-1]
    zip_path = local_dir / fn

    download_file(url, zip_path)

    if zip_path.suffix.lower() != ".zip":
        raise ValueError(f"Expected a .zip OBAT CSV package, got: {zip_path.name}")

    unzip_to_dir(zip_path, extract_dir)

    csvp = find_obat_csv(extract_dir, iso3=iso3)
    if csvp and csvp.exists():
        return csvp

    raise FileNotFoundError(f"Downloaded/unpacked but could not locate a .csv for ISO3={iso3} in {extract_dir}")


# ----------------
# DuckDB extract (CSV-only)
# ----------------
def duckdb_connect() -> duckdb.DuckDBPyConnection:
    con = duckdb.connect()
    con.execute("INSTALL spatial;")
    con.execute("LOAD spatial;")
    return con


def detect_lon_lat_cols(con: duckdb.DuckDBPyConnection, view_name: str) -> Optional[Tuple[str, str]]:
    """
    Detect lon/lat column names in the CSV.
    Adjust candidates if OBAT uses different naming.
    """
    cols = [r[0].lower() for r in con.execute(f"DESCRIBE {view_name}").fetchall()]
    lon_cands = ["lon", "long", "longitude", "x", "x_coord", "xcoord"]
    lat_cands = ["lat", "latitude", "y", "y_coord", "ycoord"]

    lon = next((c for c in lon_cands if c in cols), None)
    lat = next((c for c in lat_cands if c in cols), None)
    return (lon, lat) if lon and lat else None


def extract_obat_csv_to_aoi(
    con: duckdb.DuckDBPyConnection,
    csv_path: Path,
    aoi_geom_4326,
    out_path: Path,
    overwrite: bool = False,
) -> str:
    out_path = Path(out_path)
    ensure_parent_dir(out_path)

    if out_path.exists() and not overwrite:
        return str(out_path)

    minx, miny, maxx, maxy = aoi_geom_4326.bounds
    aoi_geojson = json.dumps(aoi_geom_4326.__geo_interface__).replace("'", "''")

    con.execute("DROP VIEW IF EXISTS obat_csv;")
    con.execute(f"CREATE VIEW obat_csv AS SELECT * FROM read_csv_auto('{str(csv_path)}');")

    lonlat = detect_lon_lat_cols(con, "obat_csv")
    if lonlat is None:
        # CSV-only pipeline cannot clip without geometry; OBAT is often attributes keyed to building IDs.
        raise RuntimeError(
            "OBAT CSV appears to have no lon/lat columns, so AOI clipping cannot be done directly.\n"
            f"CSV: {csv_path}\n\n"
            "Next step (recommended): download building footprints for the AOI (e.g., Overture buildings), "
            "then JOIN OBAT attributes to footprints on the shared building identifier fields, and THEN clip.\n"
            "If you paste the CSV column names, I’ll tell you the exact join key(s) and write the DuckDB JOIN."
        )

    lon_col, lat_col = lonlat

    sql = f"""
    COPY (
      WITH aoi AS (SELECT ST_GeomFromGeoJSON('{aoi_geojson}') AS geom)
      SELECT c.*
      FROM (
        SELECT *,
          ST_Point(CAST({lon_col} AS DOUBLE), CAST({lat_col} AS DOUBLE)) AS geom
        FROM obat_csv
      ) AS c, aoi
      WHERE
        CAST({lon_col} AS DOUBLE) BETWEEN {minx} AND {maxx}
        AND CAST({lat_col} AS DOUBLE) BETWEEN {miny} AND {maxy}
        AND ST_Intersects(c.geom, aoi.geom)
    )
    TO '{str(out_path)}' (FORMAT PARQUET);
    """
    con.execute(sql)
    return str(out_path)


# ----------------
# Pipeline runner
# ----------------
def run_pipeline_ghs_obat_csv(config_path: str | Path) -> List[str]:
    cfg = load_config(config_path)

    aoi_cfg = cfg["aoi"]
    gdf = read_aoi_geojson(aoi_cfg["path"], aoi_cfg.get("crs_out", "EPSG:4326"))
    id_col = aoi_cfg["id_col"]

    out_root = Path(cfg["output"]["root_dir"])
    overwrite = bool(cfg["output"].get("overwrite", False))
    ensure_dir(out_root)

    con = duckdb_connect()
    outputs: List[str] = []
    try:
        for _, row in gdf.iterrows():
            aoi_id = str(row[id_col])
            iso3 = get_iso3_for_row(cfg, row)

            csv_path = resolve_country_obat_csv(cfg, iso3=iso3)
            out_path = out_root / f"aoi={aoi_id}" / f"ghs_obat_{iso3}.parquet"

            out_file = extract_obat_csv_to_aoi(
                con=con,
                csv_path=csv_path,
                aoi_geom_4326=row.geometry,
                out_path=out_path,
                overwrite=overwrite,
            )
            outputs.append(out_file)
            print(f"[OK] AOI={aoi_id} ISO3={iso3} -> {out_file}")
    finally:
        con.close()

    return outputs

In [11]:
BASE = "/content/drive/MyDrive/Gates Foundation/Building Dataset Validation"
CONFIG = f"{BASE}/configs/ghs_obat.yaml"

outputs = run_pipeline_ghs_obat_csv(CONFIG)
outputs[:3], len(outputs)

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

[OK] AOI=1 ISO3=SSD -> /content/drive/MyDrive/Gates Foundation/Building Dataset Validation/outputs/ghs_obat/aoi=1/ghs_obat_SSD.parquet


(['/content/drive/MyDrive/Gates Foundation/Building Dataset Validation/outputs/ghs_obat/aoi=1/ghs_obat_SSD.parquet'],
 1)

In [10]:
from __future__ import annotations

import json
import re
import zipfile
from dataclasses import dataclass
from pathlib import Path
from typing import Dict, Iterable, List, Optional, Tuple

import duckdb
import geopandas as gpd
import pandas as pd
import requests
import yaml


# ----------------
# Config / IO
# ----------------
def load_config(path: str | Path) -> dict:
    path = Path(path)
    with path.open("r") as f:
        return yaml.safe_load(f)

def read_aoi_geojson(aoi_path: str | Path, crs_out: str = "EPSG:4326") -> gpd.GeoDataFrame:
    gdf = gpd.read_file(aoi_path)
    if gdf.crs is None:
        gdf = gdf.set_crs("EPSG:4326")
    return gdf.to_crs(crs_out)

def ensure_dir(p: Path) -> None:
    p.mkdir(parents=True, exist_ok=True)

def ensure_parent_dir(p: Path) -> None:
    p.parent.mkdir(parents=True, exist_ok=True)

def assert_is_dir_path(p: Path, cfg_key: str) -> None:
    if p.suffix:
        raise ValueError(f"{cfg_key} must be a DIRECTORY path, got file-like path: {p}")


# ----------------
# HTTP helpers
# ----------------
def download_file(url: str, dest: Path, timeout: int = 180, chunk_mb: int = 16) -> Path:
    """
    Stream download with atomic tmp write.
    """
    ensure_parent_dir(dest)
    tmp = dest.with_suffix(dest.suffix + ".part")
    if dest.exists():
        return dest

    with requests.get(url, stream=True, timeout=timeout) as r:
        r.raise_for_status()
        with tmp.open("wb") as f:
            for chunk in r.iter_content(chunk_size=chunk_mb * 1024 * 1024):
                if chunk:
                    f.write(chunk)
    tmp.replace(dest)
    return dest

def unzip_to_dir(zip_path: Path, out_dir: Path) -> None:
    ensure_dir(out_dir)
    with zipfile.ZipFile(zip_path, "r") as z:
        z.extractall(out_dir)


# ----------------
# Zenodo: world_grid.zip
# ----------------
def zenodo_file_url(record_id: int | str, filename: str) -> str:
    # Zenodo pattern works for public records
    return f"https://zenodo.org/records/{record_id}/files/{filename}?download=1"

def ensure_world_grid(cfg: dict) -> Path:
    """
    Downloads and unzips world_grid.zip; returns path to world_grid.shp
    """
    glob = cfg["globfp"]
    local_dir = Path(glob["local_dir"])
    assert_is_dir_path(local_dir, "globfp.local_dir")
    ensure_dir(local_dir)

    record_id = glob.get("zenodo_record", 15487006)
    grid_zip = local_dir / "world_grid.zip"
    grid_dir = local_dir / "world_grid"

    shp_path = grid_dir / "world_grid.shp"
    if shp_path.exists():
        return shp_path

    url = zenodo_file_url(record_id, "world_grid.zip")
    download_file(url, grid_zip, timeout=180)
    unzip_to_dir(grid_zip, grid_dir)

    if not shp_path.exists():
        # fallback: search
        cands = list(grid_dir.rglob("world_grid.shp"))
        if cands:
            return cands[0]
        raise FileNotFoundError(f"Could not locate world_grid.shp after unzip in {grid_dir}")

    return shp_path


# ----------------
# PART mapping (from data_links.txt in Zenodo record)
# ----------------
# These are the Figshare article IDs embedded in Zenodo's data_links.txt. :contentReference[oaicite:2]{index=2}
PARTS: List[Tuple[int, int, int]] = [
    (0,   400, 28879733),
    (401, 699, 28881749),
    (700, 899, 28882700),
    (900, 1299, 28889813),
    (1300, 1699, 28890593),
    (1700, 1799, 28891631),
    (1800, 1899, 28903454),
    (1900, 1999, 28903853),
    (2000, 2299, 28904453),
    (2300, 2599, 28906499),
]

def part_article_id_for_grid(grid_id: int) -> int:
    for lo, hi, aid in PARTS:
        if lo <= grid_id <= hi:
            return aid
    raise ValueError(f"grid_id {grid_id} not covered by PART ranges")


# ----------------
# Figshare API: list files + download only needed tiles
# ----------------
FIGSHARE_API = "https://api.figshare.com/v2"

@dataclass(frozen=True)
class FigshareFile:
    id: int
    name: str
    download_url: str
    size: int

def figshare_list_files(article_id: int) -> List[FigshareFile]:
    """
    Public Figshare API:
      GET /v2/articles/{article_id}
    returns JSON with 'files' each having 'download_url', 'name', 'id', 'size'
    """
    url = f"{FIGSHARE_API}/articles/{article_id}"
    r = requests.get(url, timeout=60)
    r.raise_for_status()
    meta = r.json()
    files = []
    for f in meta.get("files", []):
        files.append(
            FigshareFile(
                id=int(f["id"]),
                name=str(f["name"]),
                download_url=str(f["download_url"]),
                size=int(f.get("size", 0)),
            )
        )
    return files

def select_files_for_grid(files: List[FigshareFile], grid_id: int) -> List[FigshareFile]:
    """
    Tiles are named like:
      gridID_lon1_lat1_lon2_lat2_region1_region2.shp  (and associated .dbf/.shx/.prj)
    In Figshare, they are commonly provided as zip bundles per tile; but we support both:
      - a single .zip per tile
      - multiple shapefile component files sharing the same stem
    :contentReference[oaicite:3]{index=3}
    """
    gid = str(grid_id)

    # Prefer zip bundles containing the gridID at start of filename
    zip_hits = [f for f in files if f.name.lower().endswith(".zip") and re.match(rf"^{gid}[_\-]", f.name)]
    if zip_hits:
        return zip_hits

    # Otherwise, pull shapefile components that start with gridID_
    hits = [f for f in files if re.match(rf"^{gid}[_\-].*\.(shp|dbf|shx|prj|cpg)$", f.name, flags=re.IGNORECASE)]
    if hits:
        return hits

    # Fallback: any file containing f"{gid}_"
    hits2 = [f for f in files if f"{gid}_" in f.name]
    return hits2

def download_grid_tile(cfg: dict, grid_id: int) -> Path:
    """
    Downloads the tile file(s) for a grid_id and ensures a .shp exists locally.
    Returns path to the tile .shp.
    """
    local_dir = Path(cfg["globfp"]["local_dir"])
    tiles_dir = local_dir / "tiles"
    ensure_dir(tiles_dir)

    grid_dir = tiles_dir / f"grid_id={grid_id}"
    ensure_dir(grid_dir)

    # If already present, reuse
    shp_cands = list(grid_dir.glob("*.shp"))
    if shp_cands:
        return shp_cands[0]

    article_id = part_article_id_for_grid(grid_id)
    files = figshare_list_files(article_id)
    selected = select_files_for_grid(files, grid_id)

    if not selected:
        raise FileNotFoundError(f"No Figshare files matched grid_id={grid_id} in article {article_id}")

    # Download
    for f in selected:
        dest = grid_dir / f.name
        download_file(f.download_url, dest, timeout=600)  # tiles can be big

        # unzip if zip
        if dest.suffix.lower() == ".zip":
            unzip_to_dir(dest, grid_dir)

    # Find shapefile
    shp_cands = list(grid_dir.rglob("*.shp"))
    if not shp_cands:
        raise FileNotFoundError(f"Downloaded tile but no .shp found for grid_id={grid_id} in {grid_dir}")

    # If multiple, pick the one starting with gridID_
    gid = str(grid_id)
    shp_cands.sort(key=lambda p: (0 if p.name.startswith(gid + "_") else 1, len(p.name)))
    return shp_cands[0]


# ----------------
# Spatial: compute needed grid IDs from world_grid
# ----------------
def grid_ids_for_aoi(world_grid_shp: Path, aoi_geom_4326) -> List[int]:
    grid = gpd.read_file(world_grid_shp).to_crs("EPSG:4326")

    # Heuristic: common field names
    cand_fields = ["gridID", "grid_id", "grid_ID", "GRIDID", "GRID_ID", "id", "ID"]
    grid_field = next((c for c in cand_fields if c in grid.columns), None)
    if grid_field is None:
        raise ValueError(f"Could not find grid ID field in world_grid; columns={list(grid.columns)}")

    hits = grid[grid.intersects(aoi_geom_4326)]
    ids = sorted({int(v) for v in hits[grid_field].tolist()})
    return ids


# ----------------
# DuckDB: clip shapefile to AOI -> parquet
# ----------------
def duckdb_connect() -> duckdb.DuckDBPyConnection:
    con = duckdb.connect()
    con.execute("INSTALL spatial;")
    con.execute("LOAD spatial;")
    return con

def clip_shp_to_aoi_parquet(
    con: duckdb.DuckDBPyConnection,
    shp_path: Path,
    aoi_geom_4326,
    out_path: Path,
    overwrite: bool = False,
) -> Optional[str]:
    out_path = Path(out_path)
    ensure_parent_dir(out_path)

    if out_path.exists() and not overwrite:
        return str(out_path)

    minx, miny, maxx, maxy = aoi_geom_4326.bounds
    aoi_geojson = json.dumps(aoi_geom_4326.__geo_interface__).replace("'", "''")

    sql = f"""
    COPY (
      WITH aoi AS (SELECT ST_GeomFromGeoJSON('{aoi_geojson}') AS geom)
      SELECT b.*
      FROM ST_Read('{str(shp_path)}') AS b, aoi
      WHERE
        ST_XMin(ST_Envelope(b.geom)) <= {maxx} AND ST_XMax(ST_Envelope(b.geom)) >= {minx}
        AND ST_YMin(ST_Envelope(b.geom)) <= {maxy} AND ST_YMax(ST_Envelope(b.geom)) >= {miny}
        AND ST_Intersects(b.geom, aoi.geom)
    )
    TO '{str(out_path)}' (FORMAT PARQUET);
    """
    con.execute(sql)
    return str(out_path)


# ----------------
# Main pipeline
# ----------------
def run_pipeline_3d_globfp(config_path: str | Path) -> List[str]:
    cfg = load_config(config_path)

    aoi_cfg = cfg["aoi"]
    gdf = read_aoi_geojson(aoi_cfg["path"], aoi_cfg.get("crs_out", "EPSG:4326"))
    id_col = aoi_cfg["id_col"]

    out_root = Path(cfg["output"]["root_dir"])
    overwrite = bool(cfg["output"].get("overwrite", False))
    ensure_dir(out_root)

    world_grid_shp = ensure_world_grid(cfg)

    con = duckdb_connect()
    outputs: List[str] = []
    try:
        for _, row in gdf.iterrows():
            aoi_id = str(row[id_col])

            # 1) which tiles intersect this AOI
            grid_ids = grid_ids_for_aoi(world_grid_shp, row.geometry)
            if not grid_ids:
                print(f"[WARN] AOI={aoi_id} -> no intersecting 3D-GloBFP grids")
                continue

            # 2) download + clip each tile; write one parquet per tile, per AOI
            for grid_id in grid_ids:
                shp_path = download_grid_tile(cfg, grid_id)

                out_path = out_root / f"aoi={aoi_id}" / f"globfp_grid={grid_id}.parquet"
                out_file = clip_shp_to_aoi_parquet(
                    con=con,
                    shp_path=shp_path,
                    aoi_geom_4326=row.geometry,
                    out_path=out_path,
                    overwrite=overwrite,
                )
                if out_file:
                    outputs.append(out_file)
                    print(f"[OK] AOI={aoi_id} grid={grid_id} -> {out_file}")
    finally:
        con.close()

    return outputs

In [11]:
BASE = "/content/drive/MyDrive/Gates Foundation/Building Dataset Validation"
CONFIG = f"{BASE}/configs/3d_globfp.yaml"

outputs = run_pipeline_3d_globfp(CONFIG)
outputs[:3], len(outputs)

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

[OK] AOI=1 grid=1362 -> /content/drive/MyDrive/Gates Foundation/Building Dataset Validation/data/3d_globfp/aoi=1/globfp_grid=1362.parquet


(['/content/drive/MyDrive/Gates Foundation/Building Dataset Validation/data/3d_globfp/aoi=1/globfp_grid=1362.parquet'],
 1)

In [14]:
! ls '/content/drive/MyDrive/Gates Foundation/Building Dataset Validation/configs/'

gba.yaml		ghs_obat.yaml		      overture.yaml
ghs_obat_url_index.csv	open_buildings_temporal.yaml  ssd-juba.yaml
